In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

data_path = os.path.join("..", "data", "cleaned.csv")
df = pd.read_csv(data_path)

# Tampilkan ringkasan awal
print(df.shape)
print(df.head())
print(df.info())
print(df.isna().sum())

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [2]:
# Cek dan hapus kolom yang tidak diperlukan
# Sesuaikan nama kolom sesuai dataset
drop_cols = ["customerID"] if "customerID" in df.columns else []
df = df.drop(columns=drop_cols)


In [3]:
# Pisahkan fitur numerik dan kategorikal
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Hilangkan target dari fitur numerik jika ada
if "Churn" in numeric_cols:
    numeric_cols.remove("Churn")

In [4]:
# Imputasi nilai hilang sebelum encoding
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

if numeric_cols:
    df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

if categorical_cols:
    df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Encoding kategorikal
encoded_df = pd.DataFrame()
if categorical_cols:
    ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    encoded = ohe.fit_transform(df[categorical_cols])
    encoded_df = pd.DataFrame(
        encoded,
        columns=ohe.get_feature_names_out(categorical_cols),
        index=df.index,
    )

df_processed = pd.concat([df[numeric_cols], encoded_df], axis=1)
if "Churn" in df.columns:
    df_processed["Churn"] = df["Churn"]

# Normalisasi / scaling numerik
if numeric_cols:
    scaler = StandardScaler()
    df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

In [5]:
# Simpan hasil
processed_path = os.path.join("..", "data", "cleaned_processed.csv")
df_processed.to_csv(processed_path, index=False)

print(df_processed.head())
print("Saved processed data to", processed_path)

   SeniorCitizen    tenure  MonthlyCharges  TotalCharges  gender_Male  \
0      -0.439916 -1.277445       -1.160323     -0.994242          0.0   
1      -0.439916  0.066327       -0.259629     -0.173244          1.0   
2      -0.439916 -1.236724       -0.362660     -0.959674          1.0   
3      -0.439916  0.514251       -0.746535     -0.194766          1.0   
4      -0.439916 -1.236724        0.197365     -0.940470          0.0   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0          1.0             0.0               0.0   
1          0.0             0.0               1.0   
2          0.0             0.0               1.0   
3          0.0             0.0               0.0   
4          0.0             0.0               1.0   

   MultipleLines_No phone service  MultipleLines_Yes  ...  StreamingTV_Yes  \
0                             1.0                0.0  ...              0.0   
1                             0.0                0.0  ...              0.0   
2             

In [6]:
import joblib

joblib.dump(scaler, "../models/preprocessing.pkl"
)

['../models/preprocessing.pkl']

In [7]:
print("Mean:", scaler.mean_)
print("Scale:", scaler.scale_)

Mean: [1.62146812e-01 3.23711487e+01 6.47616925e+01 2.28191693e+03]
Scale: [3.68585436e-01 2.45577374e+01 3.00879109e+01 2.26510958e+03]
